# In-world figures

Example of generating images and showing them in VR:

<video controls src="./assets/in_world_figures.webm">

## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../systems/openmm/17-ala.openmm.zip")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: in-world figures")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)
utilities.use_transform_handles()
utilities.modes.enter_mode("move object")

## Graph generation and display

Create a moveable handle for the figure:

In [3]:
from nanover.utilities.transforms import Transform

utilities.transforms.update_transform("graph", parent="simulation", transform=Transform.from_trs(utilities.box_size * .5))
utilities.handles.update_handle("graph", parent="graph")

Add a command to generate a graph image and update it in the scene:

In [4]:
from uuid import uuid4
import matplotlib.pyplot as plt


def display_graph():
    positions = utilities.current_frame.particle_positions
    x, y, z = positions[:, 0], positions[:, 1], positions[:, 2]

    plt.scatter(x, y)
    plt.savefig("graph.png")
    plt.close()

    texture_id = f"graph.{uuid4()}"
    utilities.resources.update_resource(texture_id, type="png", path="graph.png")
    utilities.objects.update_sprite("graph", texture=texture_id, parent="graph", size=5, color=[1, 1, 1, .5])


utilities.define_command("user/graph", handler=display_graph, icon="📊", label="generate graph")